# Build geo_features_v2 + gaze_labels_v2

Consolidates three data sources into one unified geo parquet and one unified labels CSV:

| Source | Samples | Labels | Geo |
|--------|---------|--------|-----|
| GazeCapture main | ~140k | `gaze_labels.csv` | `geo_features_v1.parquet` |
| GazeCapture supplement (Keta) | ~11k | embedded in tar JSON | **extract here** |
| ETH-XGaze supplement | ~13k | `xgaze_supplement_labels.csv` | `xgaze_supplement_geo.parquet` |

**Run locally in `itracker` conda env on Mac.**

Output files:
- `geo_features_v2.parquet` → SSD root + git repo
- `gaze_labels_v2.csv` → SSD root + git repo

## 0. Setup

In [10]:
import os
import glob
import json
import tarfile
import shutil
import numpy as np
import pandas as pd
import mediapipe as mp
import webdataset as wds
from tqdm import tqdm

# ============================================================
# PATHS
# ============================================================

SSD_ROOT       = "/Volumes/Crucial X10/210"
GIT_ROOT       = "/Volumes/Crucial X10/210/eye2voice"

# Inputs
GEO_V1         = f"{SSD_ROOT}/geo_features_v1.parquet"
LABELS_V1      = f"{GIT_ROOT}/feature_engineering/gaze_labels.csv"
XGAZE_GEO      = f"{SSD_ROOT}/xgaze_supplement/xgaze_supplement_geo.parquet"
XGAZE_LABELS   = f"{SSD_ROOT}/xgaze_supplement/xgaze_supplement_labels.csv"
SUPP_TAR_DIR   = f"{SSD_ROOT}/gaze_wds_supplement/train" # where we'll download Keta's supplement tars

# Outputs
GEO_V2         = f"{SSD_ROOT}/geo_features_v2.parquet"
LABELS_V2      = f"{SSD_ROOT}/gaze_labels_v2.csv"
GEO_V2_GIT     = f"{GIT_ROOT}/feature_engineering/geo_features_v2.parquet"
LABELS_V2_GIT  = f"{GIT_ROOT}/gaze_labels_v2.csv"

os.makedirs(SUPP_TAR_DIR, exist_ok=True)
os.makedirs(os.path.dirname(GEO_V2_GIT), exist_ok=True)

print("Paths configured.")
print(f"  Supplement tars will download to: {SUPP_TAR_DIR}")
print(f"  Output geo:    {GEO_V2}")
print(f"  Output labels: {LABELS_V2}")

Paths configured.
  Supplement tars will download to: /Volumes/Crucial X10/210/gaze_wds_supplement/train
  Output geo:    /Volumes/Crucial X10/210/geo_features_v2.parquet
  Output labels: /Volumes/Crucial X10/210/gaze_labels_v2.csv


## 1. Download Keta's supplement tars from S3

5 tars, ~180MB total. Skips if already downloaded.

In [3]:
S3_SUPP = "s3://eye2voice-half-gazecapture/half-gazecapture/derived/wds_up_supplement_train_v1/train/"

existing = glob.glob(f"{SUPP_TAR_DIR}/*.tar")
if existing:
    print(f"Already have {len(existing)} tar(s) — skipping download.")
    for f in sorted(existing):
        print(f"  {os.path.basename(f)}")
else:
    print("Downloading supplement tars from S3...")
    ret = os.system(f'aws s3 sync {S3_SUPP} "{SUPP_TAR_DIR}/"')
    if ret != 0:
        raise RuntimeError("aws s3 sync failed — check credentials and region.")
    downloaded = glob.glob(f"{SUPP_TAR_DIR}/*.tar")
    print(f"Downloaded {len(downloaded)} tars.")

download: s3://eye2voice-half-gazecapture/half-gazecapture/derived/wds_up_supplement_train_v1/train/gaze-train-000000.tar to ../../gaze_wds_supplement/train/gaze-train-000000.tar
download: s3://eye2voice-half-gazecapture/half-gazecapture/derived/wds_up_supplement_train_v1/train/gaze-train-000001.tar to ../../gaze_wds_supplement/train/gaze-train-000001.tar
download: s3://eye2voice-half-gazecapture/half-gazecapture/derived/wds_up_supplement_train_v1/train/gaze-train-000002.tar to ../../gaze_wds_supplement/train/gaze-train-000002.tar
download: s3://eye2voice-half-gazecapture/half-gazecapture/derived/wds_up_supplement_train_v1/train/gaze-train-000004.tar to ../../gaze_wds_supplement/train/gaze-train-000004.tar
download: s3://eye2voice-half-gazecapture/half-gazecapture/derived/wds_up_supplement_train_v1/train/gaze-train-000003.tar to ../../gaze_wds_supplement/train/gaze-train-000003.tar
download: s3://eye2voice-half-gazecapture/half-gazecapture/derived/wds_up_supplement_train_v1/train/gaze-

## 2. Extract geo features from supplement tars

Runs MediaPipe FaceMesh on the `face.jpg` from each supplement sample.
Same exact extraction logic as `geo_features_v1.parquet`.

In [4]:
# ============================================================
# Helper: iris ratio along axis between two landmarks
# Copied verbatim from feat_eng_ckr.ipynb cell 9
# ============================================================

def ratio_along_axis(point, corner_a, corner_b):
    """Ratio of point's position along axis from corner_a to corner_b."""
    axis = corner_b - corner_a
    axis_len = np.linalg.norm(axis)
    if axis_len < 1e-6:
        return 0.5
    t = np.dot(point - corner_a, axis) / (axis_len ** 2)
    return float(np.clip(t, 0.0, 1.0))


def extract_geo_from_face(face_img_np, face_mesh):
    """
    Run MediaPipe on a face crop (H x W x 3 uint8 numpy array).
    Returns a dict of 7 features, or None if detection fails.
    
    Features (identical to geo_features_v1):
      left_iris_h     — left iris horizontal ratio
      right_iris_h    — right iris horizontal ratio
      iris_h_agreement — left minus right
      head_yaw        — nose x offset from eye center (normalized)
      head_pitch      — nose y offset below eye line (normalized)
      z_tilt          — z(chin) minus z(forehead)
      z_nose_rel      — z(nose) minus z(forehead)
    """
    h, w = face_img_np.shape[:2]
    results = face_mesh.process(face_img_np)

    if not results.multi_face_landmarks or len(results.multi_face_landmarks[0].landmark) < 478:
        return None

    lms = results.multi_face_landmarks[0].landmark

    def lm_px(idx):
        return np.array([lms[idx].x * w, lms[idx].y * h])

    # Iris horizontal ratios
    left_iris_h  = ratio_along_axis(lm_px(473), lm_px(263), lm_px(362))
    right_iris_h = ratio_along_axis(lm_px(468), lm_px(33),  lm_px(133))
    iris_h_agreement = left_iris_h - right_iris_h

    # Head pose from 2D landmarks
    nose        = lm_px(1)
    left_outer  = lm_px(263)
    right_outer = lm_px(33)
    eye_cx   = (left_outer[0] + right_outer[0]) / 2.0
    eye_cy   = (left_outer[1] + right_outer[1]) / 2.0
    eye_span = abs(left_outer[0] - right_outer[0])

    head_yaw   = (nose[0] - eye_cx) / max(eye_span, 1e-6)
    head_pitch = (nose[1] - eye_cy) / max(eye_span, 1e-6)

    # Z-depth signals
    z_tilt     = lms[152].z - lms[10].z
    z_nose_rel = lms[1].z   - lms[10].z

    return {
        'left_iris_h': left_iris_h,
        'right_iris_h': right_iris_h,
        'iris_h_agreement': iris_h_agreement,
        'head_yaw': head_yaw,
        'head_pitch': head_pitch,
        'z_tilt': z_tilt,
        'z_nose_rel': z_nose_rel,
    }

print("Extraction functions defined.")

Extraction functions defined.


In [5]:
# ============================================================
# Run extraction over all 5 supplement tars
# Expected: ~11k samples, ~30 min on M1 Mac
# ============================================================

supp_tar_urls = sorted(glob.glob(f"{SUPP_TAR_DIR}/*.tar"))
print(f"Processing {len(supp_tar_urls)} supplement tars...")

supp_geo_records = []
fail_count = 0

mp_face_mesh = mp.solutions.face_mesh

with mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,    # CRITICAL — needed for iris landmarks 468-477
    min_detection_confidence=0.5,
) as face_mesh:

    ds = wds.WebDataset(supp_tar_urls, shardshuffle=False).decode("pil")

    for sample in tqdm(ds, desc="Extracting supplement geo"):
        key = sample["__key__"]
        face_img = np.array(sample["face.jpg"])

        features = extract_geo_from_face(face_img, face_mesh)

        if features is None:
            fail_count += 1
            continue

        supp_geo_records.append({'key': key, **features})

df_supp_geo = pd.DataFrame(supp_geo_records)
print(f"\nDone!")
print(f"  Extracted: {len(df_supp_geo)}")
print(f"  Failed:    {fail_count}")
print(f"\nSample:")
print(df_supp_geo.head(3))

I0000 00:00:1775181264.633287       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Processing 6 supplement tars...


Extracting supplement geo: 17829it [02:00, 147.98it/s]


Done!
  Extracted: 17440
  Failed:    389

Sample:
            key  left_iris_h  right_iris_h  iris_h_agreement  head_yaw  \
0  00028_000782     0.472360      0.421958          0.050402  0.166175   
1  00028_000598     0.532490      0.376224          0.156266  0.173182   
2  00028_000390     0.378852      0.457789         -0.078937  0.043781   

   head_pitch   z_tilt  z_nose_rel  
0    0.266413 -0.25233   -0.340508  
1    0.272016 -0.21057   -0.326851  
2    0.406302  0.05537   -0.200979  


## 3. Build unified geo parquet (v2)

Merge: GazeCapture v1 + supplement geo + XGaze geo → `geo_features_v2.parquet`

In [6]:
GEO_COLS = ['key', 'left_iris_h', 'right_iris_h', 'iris_h_agreement',
            'head_yaw', 'head_pitch', 'z_tilt', 'z_nose_rel']

# ---- Load existing GazeCapture geo (v1) ----
df_geo_v1 = pd.read_parquet(GEO_V1)
df_geo_v1 = df_geo_v1[GEO_COLS]  # ensure column order
print(f"GazeCapture v1 geo:    {len(df_geo_v1):>7,} rows")

# ---- Supplement geo (just extracted) ----
df_supp_geo_clean = df_supp_geo[GEO_COLS]
print(f"GazeCapture supplement: {len(df_supp_geo_clean):>6,} rows")

# ---- XGaze geo ----
df_xgaze_geo = pd.read_parquet(XGAZE_GEO)
# XGaze has integer index + 'key' column — just select the right columns
df_xgaze_geo_clean = df_xgaze_geo[GEO_COLS]
print(f"XGaze geo:             {len(df_xgaze_geo_clean):>7,} rows")

# ---- Concatenate ----
df_geo_v2 = pd.concat([df_geo_v1, df_supp_geo_clean, df_xgaze_geo_clean], ignore_index=True)
print(f"\nUnified geo v2:        {len(df_geo_v2):>7,} rows")

# ---- Sanity: no duplicate keys ----
n_dupes = df_geo_v2['key'].duplicated().sum()
print(f"Duplicate keys:        {n_dupes}")
assert n_dupes == 0, "Duplicate keys found — investigate before saving."

print("\nColumn stats:")
print(df_geo_v2.describe())

GazeCapture v1 geo:    273,462 rows
GazeCapture supplement: 17,440 rows
XGaze geo:              12,976 rows

Unified geo v2:        303,878 rows
Duplicate keys:        5340


AssertionError: Duplicate keys found — investigate before saving.

In [7]:
# Find duplicate keys
duped_keys = df_geo_v2[df_geo_v2['key'].duplicated(keep=False)]['key'].unique()
print(f"Spot-checking {min(5, len(duped_keys))} duplicate keys...")

geo_cols_only = ['left_iris_h', 'right_iris_h', 'iris_h_agreement',
                 'head_yaw', 'head_pitch', 'z_tilt', 'z_nose_rel']

for key in duped_keys[:5]:
    rows = df_geo_v2[df_geo_v2['key'] == key]
    vals = rows[geo_cols_only].values
    max_diff = np.abs(vals[0] - vals[1]).max()
    print(f"  {key}: max feature diff = {max_diff:.6f}")

Spot-checking 5 duplicate keys...
  00034_000065: max feature diff = 0.000000
  00034_000158: max feature diff = 0.000000
  00034_000658: max feature diff = 0.000000
  00034_000422: max feature diff = 0.000000
  00034_000238: max feature diff = 0.000000


In [8]:
df_geo_v2 = df_geo_v2.drop_duplicates(subset='key', keep='first')
print(f"Unified geo v2 (deduped): {len(df_geo_v2):>7,} rows")
assert df_geo_v2['key'].duplicated().sum() == 0

Unified geo v2 (deduped): 298,538 rows


In [19]:
# ============================================================
# Save geo_features_v2.parquet
# ============================================================

df_geo_v2.to_parquet(GEO_V2, index=False)
print(f"Saved: {GEO_V2}")

shutil.copy(GEO_V2, GEO_V2_GIT)
print(f"Copied to git: {GEO_V2_GIT}")

Saved: /Volumes/Crucial X10/210/geo_features_v2.parquet
Copied to git: /Volumes/Crucial X10/210/eye2voice/feature_engineering/geo_features_v2.parquet


## 4. Build unified labels CSV (v2)

Merge: GazeCapture main + supplement JSON labels + XGaze labels → `gaze_labels_v2.csv`

Output schema: `key, label` — 4 classes only (Straight dropped).

Labels are always Title case: `Up`, `Down`, `Left`, `Right`.

In [11]:
VALID_CLASSES = {'Up', 'Down', 'Left', 'Right'}

# ---- Source 1: Original GazeCapture labels ----
df_labels_v1 = pd.read_csv(LABELS_V1, dtype={'subject_id': str})

# Build key, drop Straight
df_labels_v1['key'] = (
    df_labels_v1['subject_id'] + '_' +
    df_labels_v1['frame_idx'].apply(lambda x: f"{int(x):06d}")
)
df_labels_v1 = df_labels_v1[df_labels_v1['label'].isin(VALID_CLASSES)][['key', 'label']]
print(f"GazeCapture main:      {len(df_labels_v1):>7,} rows")
print(df_labels_v1['label'].value_counts())

print()

GazeCapture main:      602,467 rows
label
Down     213320
Left     188773
Right    180056
Up        20318
Name: count, dtype: int64



In [12]:
# ---- Source 2: Keta supplement — labels from tar JSONs ----
# Labels in JSON are lowercase ('up', 'down', etc.) — normalize to Title case

supp_label_records = []

for tar_path in sorted(glob.glob(f"{SUPP_TAR_DIR}/*.tar")):
    with tarfile.open(tar_path) as tf:
        for member in tf.getmembers():
            if not member.name.endswith('.json'):
                continue
            f = tf.extractfile(member)
            if f is None:
                continue
            data = json.load(f)
            key   = member.name.replace('.json', '')
            label = data['label'].capitalize()  # 'down' → 'Down'
            if label in VALID_CLASSES:
                supp_label_records.append({'key': key, 'label': label})

df_supp_labels = pd.DataFrame(supp_label_records)
print(f"GazeCapture supplement: {len(df_supp_labels):>6,} rows")
print(df_supp_labels['label'].value_counts())

print()

GazeCapture supplement: 17,829 rows
label
Up       11237
Down      2390
Left      2131
Right     2071
Name: count, dtype: int64



In [13]:
# ---- Source 3: XGaze labels ----

df_xgaze_labels = pd.read_csv(XGAZE_LABELS, dtype={'subject_id': str})

df_xgaze_labels['key'] = (
    df_xgaze_labels['subject_id'] + '_' +
    df_xgaze_labels['frame_idx'].apply(lambda x: f"{int(x):06d}")
)
df_xgaze_labels = df_xgaze_labels[df_xgaze_labels['label'].isin(VALID_CLASSES)][['key', 'label']]
print(f"XGaze:                 {len(df_xgaze_labels):>7,} rows")
print(df_xgaze_labels['label'].value_counts())

print()

XGaze:                  12,976 rows
label
Up       11476
Left       500
Right      500
Down       500
Name: count, dtype: int64



In [14]:
# ---- Concatenate all three ----

df_labels_v2 = pd.concat([df_labels_v1, df_supp_labels, df_xgaze_labels], ignore_index=True)
print(f"Unified labels v2:     {len(df_labels_v2):>7,} rows")
print()
print("Class distribution:")
print(df_labels_v2['label'].value_counts())

# ---- Sanity: no duplicate keys ----
n_dupes = df_labels_v2['key'].duplicated().sum()
print(f"\nDuplicate keys: {n_dupes}")
assert n_dupes == 0, "Duplicate keys found — investigate before saving."

# ---- Sanity: every label key has a geo entry ----
geo_keys  = set(df_geo_v2['key'].values)
label_keys = set(df_labels_v2['key'].values)
missing_geo = label_keys - geo_keys
print(f"Label keys missing from geo: {len(missing_geo)}")
if missing_geo:
    print("  Sample missing:", list(missing_geo)[:5])
    print("  (These will fall back to GEO_DEFAULT at training time — investigate if count is large.)")

Unified labels v2:     633,272 rows

Class distribution:
label
Down     216210
Left     191404
Right    182627
Up        43031
Name: count, dtype: int64

Duplicate keys: 7979


AssertionError: Duplicate keys found — investigate before saving.

In [15]:
duped_keys = df_labels_v2[df_labels_v2['key'].duplicated(keep=False)]['key'].unique()
print(f"Duplicate keys: {len(duped_keys)}")

# Check if any have conflicting labels
conflicts = []
for key in duped_keys:
    rows = df_labels_v2[df_labels_v2['key'] == key]
    if rows['label'].nunique() > 1:
        conflicts.append((key, rows['label'].tolist()))

print(f"Conflicting labels: {len(conflicts)}")
if conflicts:
    for key, labels in conflicts[:5]:
        print(f"  {key}: {labels}")

Duplicate keys: 7979
Conflicting labels: 0


In [16]:
df_labels_v2 = df_labels_v2.drop_duplicates(subset='key', keep='first')
print(f"Unified labels v2 (deduped): {len(df_labels_v2):>7,} rows")
print(df_labels_v2['label'].value_counts())
assert df_labels_v2['key'].duplicated().sum() == 0

Unified labels v2 (deduped): 625,293 rows
label
Down     214504
Left     189839
Right    181163
Up        39787
Name: count, dtype: int64


In [17]:
# ============================================================
# Save gaze_labels_v2.csv
# ============================================================

df_labels_v2.to_csv(LABELS_V2, index=False)
print(f"Saved: {LABELS_V2}")

shutil.copy(LABELS_V2, LABELS_V2_GIT)
print(f"Copied to git: {LABELS_V2_GIT}")

Saved: /Volumes/Crucial X10/210/gaze_labels_v2.csv
Copied to git: /Volumes/Crucial X10/210/eye2voice/gaze_labels_v2.csv


## 5. Final verification

In [20]:
# ============================================================
# Reload from disk and confirm everything looks right
# ============================================================

geo_check    = pd.read_parquet(GEO_V2)
labels_check = pd.read_csv(LABELS_V2)

print("=" * 50)
print("geo_features_v2.parquet")
print("=" * 50)
print(f"  Shape:   {geo_check.shape}")
print(f"  Columns: {list(geo_check.columns)}")
print(f"  Sample key: {geo_check['key'].iloc[0]}")
print()
print("=" * 50)
print("gaze_labels_v2.csv")
print("=" * 50)
print(f"  Shape:   {labels_check.shape}")
print(f"  Columns: {list(labels_check.columns)}")
print(f"  Class distribution:")
print(labels_check['label'].value_counts())
print()
print("  Sample GazeCapture key:",
      labels_check[labels_check['key'].str.startswith('0')]['key'].iloc[0])
print("  Sample XGaze key:      ",
      labels_check[labels_check['key'].str.startswith('xgaze')]['key'].iloc[0])
print()
print("All checks passed. Ready for training.")

geo_features_v2.parquet
  Shape:   (298538, 8)
  Columns: ['key', 'left_iris_h', 'right_iris_h', 'iris_h_agreement', 'head_yaw', 'head_pitch', 'z_tilt', 'z_nose_rel']
  Sample key: 00003_000000

gaze_labels_v2.csv
  Shape:   (625293, 2)
  Columns: ['key', 'label']
  Class distribution:
label
Down     214504
Left     189839
Right    181163
Up        39787
Name: count, dtype: int64

  Sample GazeCapture key: 00002_000000
  Sample XGaze key:       xgaze_0000_005796

All checks passed. Ready for training.
